<a id="inicio"></a>
    # AED 04 · Optimización con restricciones y sensibilidad

    **Núcleo U2 · S07 · RA2 / CE2.1, 2.3–2.5** · Colección docente · Edición 2026-10-08.

    Ejecute todas las celdas en orden con el entorno documentado en `../reproducibilidad/README.md`.
    Los controles necesitan kernel activo; las salidas guardadas permiten lectura. Los datos son
    casos docentes locales; las simulaciones y parámetros económicos se identifican como supuestos.
    Consulte la [guía maestra](../guia_maestra_aed.html), el
    [manual científico](../material_propio/AED_manual_cientifico.pdf) y la
    [matriz curricular](../COBERTURA.md). Los ejercicios son formativos.

### Antes de ejecutar

Núcleo · 90–120 min orientativos.

**Objetivo:** Formular y resolver una decisión factible e interpretar sensibilidad local.

**Preparación:** Ecuaciones lineales, unidades y costos de oportunidad.

**Ejemplo de referencia:** Con 100 horas y 90 de material, A=110/3 y B=80/3 producen 6800/3 UM.

**Práctica:** Cambiar horas a 110 y después a 200; comprobar cuándo deja de valer el precio sombra inicial.

**Criterio de logro:** Presentar variables, objetivo, restricciones, solución, holguras y una decisión de capacidad.

[Guía y secuencia](../guia_maestra_aed.html#ruta-aprendizaje) · [Fundamento del manual](../material_propio/AED_manual_cientifico.html#sec-optimizacion)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
RAIZ=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'_transversal/datos').is_dir())
sys.path.insert(0,str(RAIZ/'04_Analitica_Estrategica_Datos/reproducibilidad'))
from aed_recursos import (casapeumo, indicadores, boldonet, plan_produccion,
                         decision, solar, resumen_solar, pregunta, DATOS, SEMILLA)
plt.rcParams.update({'figure.figsize':(8,3.5),'axes.spines.top':False,'axes.spines.right':False})

<a id="formulacion"></a>
## 1. Variables, objetivo y región factible

Caso simulado: producir cantidades divisibles A y B con contribuciones de 40 y 30 UM por unidad.
Cada A requiere dos horas y una unidad de material; cada B, una hora y dos de material.

$$\max_{A,B\ge0}40A+30B\quad\text{s.a.}\quad2A+B\le100,\quad A+2B\le90.$$

Horas, moneda y material no se suman entre sí. La contribución es incremental y no incluye costos
fijos hundidos. Linealidad supone rendimientos constantes, aditividad y divisibilidad; una fábrica
real puede requerir lotes, cambios de turno y demanda máxima. Esas restricciones deben entrar
antes de afirmar factibilidad operativa. SciPy minimiza: negamos el objetivo, no las restricciones.

In [2]:
res=plan_produccion()
assert res.success,res.message
display(pd.Series({'A':res.x[0],'B':res.x[1],'contribucion':-res.fun}))
assert np.allclose(res.x,[110/3,80/3])
assert np.isclose(-res.fun,6800/3)
assert np.all(np.array([[2,1],[1,2]])@res.x<=np.array([100,90])+1e-8)

A                 36.666667
B                 26.666667
contribucion    2266.666667
dtype: float64

<a id="sensibilidad"></a>
## 2. Holguras y precios sombra

En el óptimo ambas restricciones están activas. El precio sombra mide cambio marginal del valor
óptimo por unidad adicional del recurso, mientras la estructura de la solución permanece vigente.
No es precio de mercado ni autorización para comprar infinitamente. En minimización del negativo,
el marginal de SciPy tiene signo opuesto al valor de contribución. Verificamos con diferencias
finitas y mostramos qué sucede al cambiar capacidades ampliamente.

In [3]:
epsilon=.01
sombra=-res.ineqlin.marginals
fd=np.array([(-plan_produccion(100+epsilon,90).fun+res.fun)/epsilon,
             (-plan_produccion(100,90+epsilon).fun+res.fun)/epsilon])
assert np.allclose(sombra,fd,atol=1e-5)
display(pd.DataFrame({'recurso':['Horas','Material'],'holgura':res.ineqlin.residual,'valor_marginal':sombra,'diferencia_finita':fd}))
@widgets.interact(horas=widgets.IntSlider(value=100,min=0,max=240,step=10),material=widgets.IntSlider(value=90,min=0,max=180,step=10))
def optimizar(horas=100,material=90):
    r=plan_produccion(horas,material)
    assert r.success
    print(f'A={r.x[0]:.2f}, B={r.x[1]:.2f}; contribución={-r.fun:.2f} UM')
    # Escala fija comparable: A <= horas/2 <= 120 y B <= material/2 <= 90.
    x=np.linspace(0,190,250)
    fig,ax=plt.subplots();ax.plot(x,horas-2*x,label='Horas');ax.plot(x,(material-x)/2,label='Material');ax.scatter(*r.x,color='#bd8841',s=80);ax.set(xlim=(0,130),ylim=(0,100),xlabel='A',ylabel='B');ax.legend();plt.show();plt.close(fig)

,recurso,holgura,valor_marginal,diferencia_finita
0,Horas,0.0,16.666667,16.666667
1,Material,0.0,6.666667,6.666667


interactive(children=(IntSlider(value=100, description='horas', max=240, step=10), IntSlider(value=90, descrip…

<a id="enteros"></a>
## 3. Proyectos indivisibles: programación entera

Si una decisión es implementar o no un proyecto, $x_i\in\{0,1\}$; redondear una solución continua
no garantiza factibilidad ni optimalidad. Para tres proyectos con beneficios [18,25,20] y costos
[10,15,12], presupuesto 25, resolvemos un MILP y contrastamos con enumeración exhaustiva.
Los beneficios son supuestos incrementales; el resultado cambia si son inciertos o solapados.

In [4]:
from scipy.optimize import milp, Bounds, LinearConstraint
from itertools import product
beneficios=np.array([18.,25.,20.]);costos=np.array([10.,15.,12.]);presupuesto=25
entero=milp(-beneficios,integrality=np.ones(3),bounds=Bounds(0,1),constraints=LinearConstraint(costos,-np.inf,presupuesto))
assert entero.success,entero.message
factibles=[np.array(x) for x in product([0,1],repeat=3) if costos@x<=presupuesto]
mejor=max(beneficios@x for x in factibles)
assert np.isclose(-entero.fun,mejor) and np.isclose(entero.x,np.round(entero.x)).all()
display(pd.Series({'seleccion':entero.x.round().tolist(),'beneficio':-entero.fun,'costo':costos@entero.x,'brecha_solver':entero.mip_gap}))

seleccion        [1.0, 1.0, 0.0]
beneficio                   43.0
costo                       25.0
brecha_solver                0.0
dtype: object

<a id="factibilidad"></a>
## 4. Inviable, no acotado y óptimo son resultados distintos

Una restricción mínima de producción puede hacer inviable el caso. Quitar recursos limitantes
puede volverlo no acotado. En ambos casos está prohibido interpretar `x` como recomendación.
El estado del solver, las holguras, la unidad y la comparación con una solución base deben quedar
en el informe. Cumplir restricciones matemáticas no valida presupuestos o capacidades mal medidos.

**Ejercicio.** ¿Pagaría 15 UM por una hora adicional con el caso base? El marginal local es
50/3 ≈ 16.67, por lo que una unidad pequeña puede convenir antes de otros costos; compruebe
la nueva solución. ¿Y 100 horas? Ya no basta el marginal local: reoptimice y compare ganancia
total con costo total. El material puede convertirse en cuello de botella.

In [5]:
from scipy.optimize import linprog
imposible=linprog([-40,-30],A_ub=[[2,1],[1,2]],b_ub=[1,1],bounds=[(10,None),(10,None)],method='highs')
sin_cota=linprog([-40,-30],bounds=[(0,None)]*2,method='highs')
assert imposible.status==2 and sin_cota.status==3
ganancia_100=-plan_produccion(200,90).fun+res.fun
print('Ganancia por 100 horas:',ganancia_100,'Costo supuesto:',1500)
assert ganancia_100<1500

Ganancia por 100 horas: 1333.3333333333335 Costo supuesto: 1500


<a id="autoevaluacion"></a>
## Autoevaluación
Seleccione y compruebe su respuesta.

In [6]:
auto_04=pregunta('¿Qué significa el precio sombra de una restricción?', ['El precio comercial garantizado del recurso', 'Un cambio marginal del óptimo dentro de su rango de validez', 'El costo fijo del proyecto'], 1, 'Es una sensibilidad local del modelo; cambios grandes requieren resolver de nuevo y revisar supuestos.')